Autoencoder Loss (MSE) The autoencoder objective requires the decoder function $g$ to fully reconstruct the input data $x$ strictly from the encoded representation $f(x)$. The objective relies on minimizing the Mean Squared Error (MSE) between the reconstruction and the original image.

Python
```
import torch
import torch.nn.functional as F

def autoencoder_loss(reconstructed_x, original_x):
    """
    Computes the MSE between the reconstruction g(f(x)) and the original input x.
    """
    return F.mse_loss(reconstructed_x, original_x)
```
Contrastive Learning Loss (with Temperature & Batching) The contrastive loss utilizes a temperature hyperparameter $\tau$ (often set to $0.07$) and minimizes the negative log probability of the positive pairs against negative pairs. To optimize performance, rather than sampling negative pairs independently, you can process a batch of $b$ positive pairs $(x, x^+)$ and compute the pairwise dot products using a single matrix multiplication.

In the resulting $b \times b$ matrix, the diagonal elements correspond to the matched positive pairs, while all off-diagonal elements represent the negative pairs.
Python
```
def contrastive_loss(z_i, z_j, temperature=0.07):
    """
    Computes the contrastive loss for a batch of l2-normalized embeddings.
    z_i: Representations of the original batch x, shape (b, d)
    z_j: Representations of the augmented batch x^+, shape (b, d)
    """
    batch_size = z_i.size(0)

    # Compute pairwise dot products (logits) using a single matrix multiplication
    # z_i is (b, d) and z_j.T is (d, b), resulting in a (b, b) logits matrix
    logits = torch.matmul(z_i, z_j.T) / temperature

    # The positive pairs are the matched elements on the diagonal
    targets = torch.arange(batch_size, device=z_i.device)

    # Cross-entropy loss automatically applies the softmax and negative log formulation
    loss = F.cross_entropy(logits, targets)

    return loss
```